# Continuous FADC3D encoder — DS OFF — default 1200/306 cache split — val every 5 epochs — seed 42

Revised default-split protocol (guide-approved).

* Split origin:      **the physical cache** — `<cache_root>/train/*.npz`
  and `<cache_root>/val/*.npz`. No RNG. No random resplit. No test partition.
* Split contract:    exactly **1200** train + **306** validation patients.
* Model:             `unet3d_fadc_continuous_encoder`  (arch_kind: `continuous`)
* Deep supervision:  **False**
* Formal validation: every **5 epochs**, all 306 val cases,
  sliding-window at overlap 0.5, sw_batch_size 4.
  **Measured wall-time**: ~123.4 min per pass on Kaggle GPU (T4/P100),
  so 20 passes over 100 epochs adds ~41 h to the training budget.
  Plan Kaggle sessions accordingly and rely on the cross-session
  resume pathway.
* Checkpoints:       `last_checkpoint.pth` atomically every epoch;
  `checkpoint_epochNNN.pth` every 5 epochs.
* Resume safety:     rejects any 70/10/20 checkpoint, any DS/noDS mismatch,
  and any discrete/continuous mismatch.
* No final-test cell — this protocol reports formal validation on the
  fixed 306-patient default validation set only.

Continuous FADC3D encoder. Deep supervision is not supported for this architecture. The trainer natively dispatches on model name — no runtime monkey-patch adapter is used.


In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────
# Default-cache 1200/306 partition. No RNG. No test partition. Seed 42 is
# still used for model init, augmentation, and DataLoader reproducibility.
SEED                = 42
GIT_BRANCH          = "feature/fadc3d-continuous-adadr"

# EXPECTED_GIT_COMMIT must be pinned before running — an empty value is a
# preflight hard-fail. Set to the exact commit that carries this notebook +
# the default-cache snapshot API + the native continuous dispatch.
EXPECTED_GIT_COMMIT = "54892316f8ceff2d48a7632923556495cb7ff8ff"   # <-- PIN THIS BEFORE LAUNCH (40-hex SHA)

# NOTE: the Kaggle dataset owner slug is 'bharathkumarvemuri' for this run.
DATA_ROOT              = "/kaggle/input/datasets/bharathkumarvemuri/mama-mia-preprocessed-cache-2ch"
PREPROCESSED_CACHE_DIR = DATA_ROOT
CODE_DIR               = "/kaggle/working/FADC-3D"

# Fresh output directory. Cannot collide with any 70/10/20 or legacy run.
OUTPUT_DIR_SMOKE  = "/kaggle/working/outputs/fadc3d_continuous_encoder_nods_defaultsplit_val5_s42_smoke"
OUTPUT_DIR_FULL   = "/kaggle/working/outputs/fadc3d_continuous_encoder_nods_defaultsplit_val5_s42"

# Default-cache snapshot lives under the writable output dir on Kaggle.
MANIFEST_CSV      = f"{OUTPUT_DIR_FULL}/default_cache_train_val_manifest.csv"
MANIFEST_META     = f"{OUTPUT_DIR_FULL}/default_cache_train_val_manifest_metadata.json"

# Full-training hyperparameters (UNCHANGED vs previous corrected runs).
EPOCHS         = 100
BATCH_SIZE     = 2
NUM_WORKERS    = 4
PATCH_SIZE     = [128, 128, 64]
LEARNING_RATE  = 1e-4
WARMUP_EPOCHS  = 5

# THE ABLATION KNOB FOR THIS NOTEBOOK.
DEEP_SUPERVISION = False

# Default-cache contract — enforced by the snapshot API and by preflight.
EXPECTED_TRAIN_COUNT = 1200
EXPECTED_VAL_COUNT   = 306
EXPECTED_TOTAL_COUNT = EXPECTED_TRAIN_COUNT + EXPECTED_VAL_COUNT   # 1506

# k_att schedule (UNCHANGED — inert for continuous).
K_ATT_TEMP_START    = 2.0
K_ATT_TEMP_END      = 1.0
K_ATT_ANNEAL_EPOCHS = 60

# Attention diversity aux DISABLED. Position attention DISABLED.
ATTN_DIVERSITY_WEIGHT = 0.0
USE_POSITION_ATT      = False

# Smoke-test parameters.
SMOKE_PATCH_SIZE = [48, 48, 32]

MODEL_NAME = "unet3d_fadc_continuous_encoder"
ARCH_KIND  = "continuous"

# ── VALIDATION SCHEDULE (formal only) ──────────────────────────────────
# Guide-approved cadence: full formal validation every 5 epochs.
# Measured wall-time on Kaggle GPU (T4/P100): ~123.4 min per pass across
# all 306 val cases at overlap 0.5. Over 100 epochs that's 20 passes ->
# ~41 h of validation on top of training. Plan Kaggle sessions and
# checkpoint downloads accordingly.
VAL_EVERY           = 5     # formal validation every 5 epochs
VAL_OVERLAP         = 0.5   # canonical evaluation overlap
VAL_SW_BATCH_SIZE   = 4
CHECKPOINT_EVERY    = 5     # named checkpoint snapshot every 5 epochs

# ── RESUME (cross-session Kaggle) ──────────────────────────────────────
# Leave BOTH empty for fresh training (default). When resuming from a
# previous Kaggle session:
#   RESUME_INPUT_DIR = "/kaggle/input/<your-uploaded-dataset-slug>"
#   RESUME_FROM      = "last_checkpoint.pth"
# The default-cache snapshot cell verifies arch_identity + DS + split
# identity (including split_kind='default_cache' and the manifest SHA)
# match this notebook's CONFIG before copying the ckpt into place.
RESUME_INPUT_DIR = ""
RESUME_FROM      = ""

# ── EXPECTED_MANIFEST_SHA256 (paste-across-notebooks) ──────────────────
# The default-cache snapshot is fully determined by the physical cache
# contents, so the DS notebook's SHA should equal the noDS notebooks'.
# Leaving this empty accepts whatever SHA the snapshot cell computes;
# pasting a value here promotes the check to a hard preflight equality
# — the recommended mode once the DS run has generated the canonical SHA.
EXPECTED_MANIFEST_SHA256 = "a807fcc35f2423ef39227d297b2e7ee0f1bb2ffb6725f50077359f8c5b9ea3e6"   # 64-hex or empty

print(f"SEED                : {SEED}")
print(f"BRANCH              : {GIT_BRANCH}")
print(f"MODEL_NAME          : {MODEL_NAME}")
print(f"ARCH_KIND           : {ARCH_KIND}")
print(f"OUTPUT_DIR_FULL     : {OUTPUT_DIR_FULL}")
print(f"OUTPUT_DIR_SMOKE    : {OUTPUT_DIR_SMOKE}")
print(f"MANIFEST_CSV        : {MANIFEST_CSV}")
print(f"DEEP_SUPERVISION    : {DEEP_SUPERVISION}")
print(f"CACHE CONTRACT      : {EXPECTED_TRAIN_COUNT} train + {EXPECTED_VAL_COUNT} val (no test)")
print(f"PATCH_SIZE (train)  : {PATCH_SIZE}")
print(f"EPOCHS / BATCH      : {EPOCHS} / {BATCH_SIZE}")
print()
print("VALIDATION SCHEDULE (formal only, guide-approved cadence)")
print(f"  validation cases    : all 306 default-cache val patients")
print(f"  formal overlap      : {VAL_OVERLAP}")
print(f"  validation frequency: every {VAL_EVERY} epochs")
print(f"  sw_batch_size       : {VAL_SW_BATCH_SIZE}")
print(f"  checkpoint_every    : {CHECKPOINT_EVERY}")
print()
print(f"EXPECTED_GIT_COMMIT : {EXPECTED_GIT_COMMIT or '(empty — preflight will refuse to run)'}")
print(f"EXPECTED_MANIFEST_SHA256 : {EXPECTED_MANIFEST_SHA256 or '(empty — will accept whatever snapshot yields)'}")


In [ ]:
# ── PREFLIGHT — revised default-split contract ─────────────────────────
# Static assertions that must hold before any expensive cell runs.
import os, re, sys

# 1) EXPECTED_GIT_COMMIT must be pinned (revised protocol requirement).
if not EXPECTED_GIT_COMMIT:
    raise SystemExit(
        "PREFLIGHT: EXPECTED_GIT_COMMIT is empty. Pin the exact source "
        "commit (40-hex) in the CONFIG cell before launching this notebook."
    )
if not re.fullmatch(r"[0-9a-fA-F]{40}", EXPECTED_GIT_COMMIT):
    raise SystemExit(f"PREFLIGHT: EXPECTED_GIT_COMMIT is not a 40-hex SHA: {EXPECTED_GIT_COMMIT!r}")

# 2) VAL_EVERY / CHECKPOINT_EVERY / VAL_OVERLAP — guide-approved contract.
assert VAL_EVERY == 5,           f"PREFLIGHT: VAL_EVERY must be 5 (got {VAL_EVERY})"
assert CHECKPOINT_EVERY == 5,    f"PREFLIGHT: CHECKPOINT_EVERY must be 5 (got {CHECKPOINT_EVERY})"
assert abs(VAL_OVERLAP - 0.5) < 1e-9, f"PREFLIGHT: VAL_OVERLAP must be 0.5 (got {VAL_OVERLAP})"
assert VAL_SW_BATCH_SIZE == 4,   f"PREFLIGHT: VAL_SW_BATCH_SIZE must be 4 (got {VAL_SW_BATCH_SIZE})"

# 2b) Patch-size divisibility contract. The FADC-3D UNet has 4 pool
#     stages, so every spatial dim must be a positive multiple of 16.
#     Catching it here (before dependency install / clone / training)
#     avoids the deep torch.cat mismatch that surfaces only in dec4.
def _assert_patch_div16(name, ps):
    _bad = [(i, v) for i, v in enumerate(ps) if v <= 0 or (v % 16) != 0]
    if _bad:
        _off = ', '.join(f'dim {i}={v}' for i, v in _bad)
        raise SystemExit(
            f'PREFLIGHT: invalid {name}={list(ps)!r}: {_off}. '
            f'This UNet has 4 pool stages; every spatial dimension must '
            f'be a positive multiple of 16 (e.g. round 24 -> 32 or 16).'
        )
_assert_patch_div16('PATCH_SIZE',       PATCH_SIZE)
_assert_patch_div16('SMOKE_PATCH_SIZE', SMOKE_PATCH_SIZE)


# 3) DEEP_SUPERVISION matches this notebook's contract.
assert DEEP_SUPERVISION is False, (
    f"PREFLIGHT: DEEP_SUPERVISION must be False for this notebook "
    f"(got {DEEP_SUPERVISION!r})"
)

# 4) Model / arch kind sanity.
assert MODEL_NAME == "unet3d_fadc_continuous_encoder", MODEL_NAME
assert ARCH_KIND  == "continuous", ARCH_KIND

# 5) Output directory contains 'defaultsplit_val5' and never collides with
#    a legacy run.
assert "defaultsplit_val5" in OUTPUT_DIR_FULL, \
    f"PREFLIGHT: OUTPUT_DIR_FULL must contain 'defaultsplit_val5' (got {OUTPUT_DIR_FULL!r})"
for _legacy in ("/outputs/fadc3d_correct_encoder_s42",
                "/outputs/fadc3d_correct_encoder_nods_s42",
                "/outputs/fadc3d_correct_encoder_ds_split701020_s42",
                "/outputs/fadc3d_correct_encoder_nods_split701020_s42"):
    assert OUTPUT_DIR_FULL != _legacy, \
        f"PREFLIGHT: OUTPUT_DIR_FULL collides with a legacy run: {_legacy}"

# 6) Default-cache contract counts.
assert EXPECTED_TRAIN_COUNT == 1200 and EXPECTED_VAL_COUNT == 306, \
    f"PREFLIGHT: default-cache contract is 1200/306, got {EXPECTED_TRAIN_COUNT}/{EXPECTED_VAL_COUNT}"

# 7) EXPECTED_MANIFEST_SHA256, if set, is 64-hex.
if EXPECTED_MANIFEST_SHA256:
    assert re.fullmatch(r"[0-9a-fA-F]{64}", EXPECTED_MANIFEST_SHA256), \
        f"PREFLIGHT: EXPECTED_MANIFEST_SHA256 must be empty or 64-hex, got {EXPECTED_MANIFEST_SHA256!r}"

# 8) Resume-mode derivation.
if bool(RESUME_INPUT_DIR) != bool(RESUME_FROM):
    if RESUME_INPUT_DIR and not RESUME_FROM:
        RESUME_FROM = "last_checkpoint.pth"
        print(f"PREFLIGHT: RESUME_FROM defaulted to {RESUME_FROM!r}")
    else:
        raise SystemExit(
            f"PREFLIGHT: RESUME_INPUT_DIR and RESUME_FROM must both be set "
            f"(resume) or both empty (fresh). "
            f"Got RESUME_INPUT_DIR={RESUME_INPUT_DIR!r} RESUME_FROM={RESUME_FROM!r}."
        )
IS_RESUME_MODE = bool(RESUME_INPUT_DIR) and bool(RESUME_FROM)
RESUME_SRC_PATH = os.path.join(RESUME_INPUT_DIR, RESUME_FROM) if IS_RESUME_MODE else ""
if IS_RESUME_MODE:
    if not os.path.isdir(RESUME_INPUT_DIR):
        raise SystemExit(f"PREFLIGHT: RESUME_INPUT_DIR does not exist: {RESUME_INPUT_DIR}")
    if not os.path.isfile(RESUME_SRC_PATH):
        raise SystemExit(f"PREFLIGHT: resume checkpoint not found: {RESUME_SRC_PATH}")
    print(f"PREFLIGHT: RESUME MODE  src={RESUME_SRC_PATH}")
else:
    print("PREFLIGHT: FRESH MODE  (no resume ckpt configured)")

# 9) Fresh-only guard on OUTPUT_DIR_FULL. In resume mode we allow the
#    expected copies (last_checkpoint.pth + periodic snapshots); in fresh
#    mode we refuse any .pth to prevent silent cross-run pollution.
_stale = []
if os.path.isdir(OUTPUT_DIR_FULL):
    for name in os.listdir(OUTPUT_DIR_FULL):
        if name.endswith(".pth"):
            _stale.append(os.path.join(OUTPUT_DIR_FULL, name))
if _stale and not IS_RESUME_MODE:
    raise SystemExit(
        "PREFLIGHT: OUTPUT_DIR_FULL already contains checkpoint files:\n  "
        + "\n  ".join(_stale) +
        "\n\nThis run is fresh-only. Point OUTPUT_DIR_FULL at a clean directory."
    )
if _stale and IS_RESUME_MODE:
    _allowed = {os.path.basename(RESUME_FROM)}
    _bad = [p for p in _stale
            if os.path.basename(p) not in _allowed
            and not os.path.basename(p).startswith("checkpoint_epoch")]
    if _bad:
        raise SystemExit(
            "PREFLIGHT: OUTPUT_DIR_FULL contains .pth files not part of the "
            f"resume copy set (allowed: {sorted(_allowed)!r} + checkpoint_epoch*.pth):\n  "
            + "\n  ".join(_bad)
        )

print("PREFLIGHT OK")
print(f"  MODEL_NAME             : {MODEL_NAME}")
print(f"  ARCH_KIND              : {ARCH_KIND}")
print(f"  DEEP_SUPERVISION       : {DEEP_SUPERVISION}")
print(f"  VAL_EVERY              : {VAL_EVERY}   (every-5 formal)")
print(f"  CHECKPOINT_EVERY       : {CHECKPOINT_EVERY}")
print(f"  default-cache contract : {EXPECTED_TRAIN_COUNT} train + {EXPECTED_VAL_COUNT} val (no test)")
print(f"  EXPECTED_GIT_COMMIT    : {EXPECTED_GIT_COMMIT}")
print(f"  EXPECTED_MANIFEST_SHA  : {EXPECTED_MANIFEST_SHA256 or '(any)'}")


In [ ]:
# ── 1. INSTALL DEPS + REQUIRE CUDA ─────────────────────────────────────
# MONAI 1.5.2 is the pinned version. Rationale:
#   - 1.4.0 (previous pin) is Python-3.12 clean but its metadata caps
#     numpy at <2.0. Kaggle ships numpy 2.x, so `pip install monai==1.4.0`
#     silently downgrades numpy and cascade-breaks torch / skimage.
#   - 1.5.0+ dropped that upper cap; 1.5.2 is the latest 1.5.x point
#     release and keeps the same transform API this notebook uses.
#   - Older MONAI 1.3.x used importer.find_module() which was removed in
#     Python 3.12 -- do not roll back further than 1.4.
#
# Defensive install pattern:
#   1. Uninstall whatever MONAI the Kaggle base image ships (usually 1.3.x).
#   2. Install the pinned version fresh from PyPI.
#   3. Assert the loaded version matches. If it doesn't, the session
#      already imported the wrong MONAI into sys.modules — force a
#      Factory-Reset restart of the kernel (Kaggle: 'Session' ->
#      'Factory Reset & Reload'), then re-run.
import subprocess, sys, torch

subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "-q", "monai"],
    check=False,   # tolerate 'not installed'
)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "monai[nibabel,skimage,pillow,tensorboard]==1.5.2", "einops"])

_MONAI_PIN = "1.5.2"
try:
    import monai
except Exception as e:
    raise SystemExit(f"MONAI import failed after install: {e!r}")
if getattr(monai, "__version__", None) != _MONAI_PIN:
    raise SystemExit(
        f"MONAI version guard failed: expected {_MONAI_PIN}, "
        f"got {getattr(monai, '__version__', 'unknown')}. "
        "The Kaggle kernel has cached a different MONAI in sys.modules. "
        "Fix: Kaggle top bar -> 'Session' -> 'Factory Reset & Reload', "
        "then re-run this cell from the top."
    )
print(f"monai      : {monai.__version__}")

import numpy as _np
_np_major = int(_np.__version__.split('.')[0])
if _np_major < 2:
    raise SystemExit(
        f"numpy guard: expected numpy>=2, got {_np.__version__}. "
        "The MONAI install just downgraded numpy, which will cascade-break "
        "torch/skimage. Fix: Kaggle top bar -> 'Session' -> 'Factory Reset "
        "& Reload', then re-run this cell."
    )
print(f"numpy      : {_np.__version__}")

assert torch.cuda.is_available(), "This notebook requires a CUDA GPU."
print(f"torch      : {torch.__version__}")
print(f"cuda       : {torch.version.cuda}")
print(f"device     : {torch.cuda.get_device_name(0)}")
print(f"vram (GB)  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}")


In [ ]:
# ── 2. CLONE / CHECKOUT AT PINNED COMMIT ───────────────────────────────
# Detaches HEAD at EXPECTED_GIT_COMMIT. Never runs `git pull` afterwards —
# the pin is the ground truth for reproducibility.
import os, subprocess

os.makedirs(os.path.dirname(CODE_DIR), exist_ok=True)
if not os.path.isdir(CODE_DIR):
    subprocess.check_call([
        "git", "clone", "--branch", GIT_BRANCH, "--single-branch",
        "https://github.com/Vemuri-BK/FADC-3D.git", CODE_DIR,
    ])
    # If clone URL differs on your account, edit the line above.
subprocess.check_call(["git", "-C", CODE_DIR, "fetch", "--all", "--tags"])
subprocess.check_call(["git", "-C", CODE_DIR, "checkout", "--detach", EXPECTED_GIT_COMMIT])
head = subprocess.check_output(
    ["git", "-C", CODE_DIR, "rev-parse", "HEAD"]
).decode().strip()
assert head == EXPECTED_GIT_COMMIT, f"git rev-parse HEAD != EXPECTED_GIT_COMMIT ({head} vs {EXPECTED_GIT_COMMIT})"
print(f"CODE_DIR : {CODE_DIR}")
print(f"HEAD     : {head}   (pinned)")


In [ ]:
# ── 3. DEFAULT-CACHE SNAPSHOT — 1200 train + 306 val, no RNG, no test ─
# Reads the read-only .npz cache in place; writes only the manifest CSV +
# metadata JSON under OUTPUT_DIR_FULL. Never copies / moves the .npz files.
# The split for every patient is defined by its physical subdir; no random
# reassignment ever happens.
import os, sys, json
sys.path.insert(0, CODE_DIR)

from training.split_manifest import (
    COLLECTIONS,
    generate_default_snapshot,
    verify_default_manifest_partitions,
    manifest_sha256,
    load_manifest,
)

os.makedirs(OUTPUT_DIR_FULL, exist_ok=True)

# If a snapshot already exists (idempotent re-run), trust it. This snapshot
# is a contract — regenerating would invalidate every downstream ckpt's
# split_identity. Otherwise, build it fresh from the cache.
if os.path.exists(MANIFEST_CSV):
    print(f"MANIFEST_CSV already exists: {MANIFEST_CSV}")
    print("  -> not regenerating; existing snapshot is authoritative.")
    with open(MANIFEST_META, encoding="utf-8") as f:
        meta = json.load(f)
else:
    meta = generate_default_snapshot(
        cache_root=PREPROCESSED_CACHE_DIR,
        csv_path=MANIFEST_CSV,
        meta_path=MANIFEST_META,
        expected_train=EXPECTED_TRAIN_COUNT,
        expected_val=EXPECTED_VAL_COUNT,
    )

# Belt-and-braces: verify the CSV on disk one more time.
partition_summary = verify_default_manifest_partitions(
    MANIFEST_CSV,
    expected_train=EXPECTED_TRAIN_COUNT,
    expected_val=EXPECTED_VAL_COUNT,
)
MANIFEST_SHA256 = manifest_sha256(MANIFEST_CSV)
assert MANIFEST_SHA256 == meta["csv_sha256"], "metadata SHA256 does not match CSV on disk!"

# If the CONFIG cell pinned an EXPECTED_MANIFEST_SHA256, require exact match.
if EXPECTED_MANIFEST_SHA256:
    if EXPECTED_MANIFEST_SHA256.lower() != MANIFEST_SHA256.lower():
        raise SystemExit(
            f"MANIFEST_SHA256 mismatch:\n"
            f"  computed          : {MANIFEST_SHA256}\n"
            f"  EXPECTED (CONFIG) : {EXPECTED_MANIFEST_SHA256}\n"
            f"Either the mounted cache differs from the pinned one, or "
            f"EXPECTED_MANIFEST_SHA256 was mistyped."
        )
    print(f"MANIFEST_SHA256 matches EXPECTED_MANIFEST_SHA256 : {MANIFEST_SHA256}")

# Load partitions and re-assert every guarantee the tests prove.
train_cases = load_manifest(MANIFEST_CSV, split="train", cache_root=PREPROCESSED_CACHE_DIR)
val_cases   = load_manifest(MANIFEST_CSV, split="val",   cache_root=PREPROCESSED_CACHE_DIR)

n_train = partition_summary["n_train"]
n_val   = partition_summary["n_val"]
n_test  = partition_summary["n_test"]

assert n_train == EXPECTED_TRAIN_COUNT == 1200, n_train
assert n_val   == EXPECTED_VAL_COUNT   == 306,  n_val
assert n_test  == 0, f"expected 0 test rows, got {n_test}"

train_ids = {c["patient_id"] for c in train_cases}
val_ids   = {c["patient_id"] for c in val_cases}
assert train_ids.isdisjoint(val_ids), "train/val overlap detected"
assert len(train_ids) + len(val_ids) == EXPECTED_TOTAL_COUNT

# Every referenced .npz exists (load_manifest already checks, re-assert).
for c in train_cases + val_cases:
    assert os.path.exists(c["npz_path"]), c["npz_path"]

print("DEFAULT-CACHE SNAPSHOT OK")
print(f"  split_kind          : default_cache (no RNG)")
print(f"  total patients      : {EXPECTED_TOTAL_COUNT}")
print(f"  train / val / test  : {n_train} / {n_val} / {n_test}")
print(f"  manifest CSV        : {MANIFEST_CSV}")
print(f"  manifest metadata   : {MANIFEST_META}")
print()
print("=" * 70)
print("!! DEFAULT-CACHE MANIFEST SHA256 — the same SHA is expected in")
print("!! every notebook of this three-way ablation.")
print(f"    {MANIFEST_SHA256}")
print("=" * 70)
print()
print("Per-collection counts:")
for coll in COLLECTIONS:
    ps = meta["n_per_collection_split"].get(coll, {})
    n_c = sum(ps.values())
    if n_c:
        print(f"  {coll:6s}  n={n_c:4d}  train={ps.get('train',0):4d}  "
              f"val={ps.get('val',0):3d}  test={ps.get('test',0):4d}")
print()
print("5 example patient_ids per split:")
for name, ids in (("train", sorted(train_ids)), ("val", sorted(val_ids))):
    print(f"  {name:5s}  {ids[:5]}")

# ── RESUME MODE: verify + copy the resume ckpt into OUTPUT_DIR_FULL ────
if IS_RESUME_MODE:
    import shutil, torch
    print()
    print("=" * 70)
    print(f"RESUME: loading source ckpt from {RESUME_SRC_PATH}")
    _r_ckpt = torch.load(RESUME_SRC_PATH, map_location="cpu", weights_only=False)
    _r_arch  = _r_ckpt.get("arch_identity",  {}) or {}
    _r_split = _r_ckpt.get("split_identity", {}) or {}
    _r_epoch_completed = int(_r_ckpt.get("epoch", -1))

    print(f"  source completed ep   : {_r_epoch_completed}")
    print(f"  source best_dice      : {float(_r_ckpt.get('best_dice', 0.0)):.4f}")
    print(f"  source arch_identity  : {_r_arch}")
    print(f"  source split_identity : {_r_split}")

    if _r_arch.get("model_name") != MODEL_NAME:
        raise SystemExit(f"RESUME: model_name mismatch (ckpt={_r_arch.get('model_name')!r} vs CONFIG={MODEL_NAME!r})")
    if _r_arch.get("arch_kind", "discrete") != ARCH_KIND:
        raise SystemExit(f"RESUME: arch_kind mismatch (ckpt={_r_arch.get('arch_kind')!r} vs CONFIG={ARCH_KIND!r})")
    if bool(_r_arch.get("deep_supervision", False)) != bool(DEEP_SUPERVISION):
        raise SystemExit(f"RESUME: deep_supervision mismatch (ckpt={_r_arch.get('deep_supervision')!r} vs CONFIG={DEEP_SUPERVISION!r})")
    if _r_split.get("split_kind") != "default_cache":
        raise SystemExit(
            f"RESUME: split_kind mismatch — the ckpt is not from a default-cache run "
            f"(ckpt={_r_split.get('split_kind')!r}). 70/10/20 checkpoints cannot be "
            f"resumed under this default-cache notebook."
        )
    if _r_split.get("split_manifest_sha256") != MANIFEST_SHA256:
        raise SystemExit(
            f"RESUME: split_manifest_sha256 mismatch "
            f"(ckpt={_r_split.get('split_manifest_sha256')!r} vs manifest={MANIFEST_SHA256!r})"
        )
    if _r_epoch_completed + 1 >= EPOCHS:
        raise SystemExit(f"RESUME: nothing to do — ckpt already completed epoch {_r_epoch_completed+1}/{EPOCHS}")

    # Copy the ckpt + any companion files into OUTPUT_DIR_FULL.
    _dst = os.path.join(OUTPUT_DIR_FULL, os.path.basename(RESUME_FROM))
    shutil.copy2(RESUME_SRC_PATH, _dst)
    for _companion in ("train_log.json", "meta.json"):
        _src_c = os.path.join(RESUME_INPUT_DIR, _companion)
        if os.path.isfile(_src_c):
            shutil.copy2(_src_c, os.path.join(OUTPUT_DIR_FULL, _companion))
    for _snap in sorted(os.listdir(RESUME_INPUT_DIR)):
        if _snap.startswith("checkpoint_epoch") and _snap.endswith(".pth"):
            shutil.copy2(os.path.join(RESUME_INPUT_DIR, _snap),
                         os.path.join(OUTPUT_DIR_FULL, _snap))
    print(f"RESUME: copied to {_dst}")
    print("=" * 70)


In [ ]:
# ── 4. ARCHITECTURE VERIFIER + CORRECTNESS TESTS ──────────────────────
import os, sys, subprocess

# Verify the model factory produces the expected block count.
sys.path.insert(0, CODE_DIR)
if ARCH_KIND == "continuous":
    from models.unet_3d_fadc_continuous import (
        build_unet3d_fadc_continuous, EXPECTED_ADAPTIVE_BLOCK_COUNT as _EXP_C,
    )
    _m = build_unet3d_fadc_continuous(
        model_name=MODEL_NAME, in_channels=2, out_channels=2,
        base_filters=32, deep_supervision=False,
    )
    _n = _m.count_adaptive_blocks()
    _exp = _EXP_C["encoder"]
    print(f"[verifier] {MODEL_NAME}: adaptive_blocks={_n} / expected={_exp}")
    assert _n == _exp, (_n, _exp)
    del _m
else:
    from models.unet_3d_fadc_correct import (
        build_unet3d_fadc_correct, EXPECTED_ADAPTIVE_CONV_COUNT as _EXP_D,
    )
    _m = build_unet3d_fadc_correct(
        model_name=MODEL_NAME, in_channels=2, out_channels=2,
        base_filters=32, deep_supervision=DEEP_SUPERVISION,
        adakern_cfg={"use_position_att": USE_POSITION_ATT},
    )
    _n = _m.count_adaptive_convs()
    _placement = MODEL_NAME.split("_")[2]
    _exp = _EXP_D[_placement]
    print(f"[verifier] {MODEL_NAME}: placement={_placement}  adaptive_convs={_n} / expected={_exp}")
    assert _n == _exp, (_n, _exp)
    del _m

# Run the correctness test suites the guide-approved run depends on.
_tests_to_run = [
    "tests/test_split_manifest.py",
    "tests/test_default_split.py",
    "tests/test_patch_validation.py",
]
if ARCH_KIND == "continuous":
    _tests_to_run.append("tests/test_fadc_3d_continuous.py")
else:
    _tests_to_run.append("tests/test_fadc_3d_correct.py")

for rel in _tests_to_run:
    path = os.path.join(CODE_DIR, rel)
    print(f"---- {rel} ----")
    r = subprocess.run([sys.executable, path], cwd=CODE_DIR,
                       capture_output=True, text=True)
    print(r.stdout[-4000:])
    if r.returncode != 0:
        sys.stderr.write(r.stderr[-2000:])
        raise SystemExit(f"{rel} FAILED")
    print(f"---- {rel} PASSED ----\n")
print("All correctness tests PASSED.")


In [ ]:
# ── 5b. CONTINUOUS SMOKE + GPU BENCHMARK (mandatory before FULL train) ─
# Continuous-notebook only. Writes OUTPUT_DIR_FULL/continuous_smoke_ok.json
# on success; the full-training cell (6) refuses to run without it.
#
# Two things happen here:
#   (a) GPU BENCHMARK — one fwd+bwd+step at the production batch/patch
#       so we know the peak VRAM footprint and per-step wall time up front.
#       If this OOMs or overruns, the run is not fit for this Kaggle GPU.
#   (b) 2-EPOCH SMOKE — the trainer itself, --smoke_test path, against the
#       real preprocessed cache + default-cache manifest. Proves the whole
#       pipeline (loader + augment + AMP + continuous fwd/bwd + val +
#       atomic ckpt) runs end-to-end on this specific machine before we
#       commit to the 100-ep budget.
import os, sys, json, gc, time, subprocess
import torch

marker_path = os.path.join(OUTPUT_DIR_FULL, "continuous_smoke_ok.json")

# ── (a) GPU BENCHMARK ──────────────────────────────────────────────────
sys.path.insert(0, CODE_DIR)
from models.unet_3d_fadc_continuous import build_unet3d_fadc_continuous

if not torch.cuda.is_available():
    raise SystemExit("CONTINUOUS GATE: CUDA required for this benchmark.")

torch.cuda.empty_cache(); gc.collect(); torch.cuda.reset_peak_memory_stats()
_model = build_unet3d_fadc_continuous(
    MODEL_NAME, in_channels=2, out_channels=2, base_filters=32,
    deep_supervision=False,
).cuda().train()
_x = torch.randn(BATCH_SIZE, 2, *PATCH_SIZE, device="cuda")
_opt = torch.optim.SGD(_model.parameters(), lr=1e-5)
_scaler = torch.amp.GradScaler("cuda")

torch.cuda.synchronize(); _t0 = time.time()
with torch.amp.autocast("cuda", dtype=torch.float16):
    _y = _model(_x); _loss = _y.float().pow(2).mean()
_scaler.scale(_loss).backward()
_scaler.step(_opt); _scaler.update()
torch.cuda.synchronize()
_step_s = time.time() - _t0
_peak_mb = torch.cuda.max_memory_allocated() / 1e6
_gpu_total_mb = torch.cuda.get_device_properties(0).total_memory / 1e6
print(f"(a) train step  : {_step_s*1e3:6.1f} ms  peak {_peak_mb:6.0f} MB "
      f"(headroom {_gpu_total_mb - _peak_mb:.0f} MB of {_gpu_total_mb:.0f} MB)")

# Bench pass/fail. Choose conservatively — we want to catch OOM on this
# specific mounted GPU before committing to the full 100-ep run.
_headroom_mb = _gpu_total_mb - _peak_mb
if _headroom_mb < 500:
    raise SystemExit(
        f"CONTINUOUS GATE: peak train-step VRAM ({_peak_mb:.0f} MB) leaves only "
        f"{_headroom_mb:.0f} MB headroom on a {_gpu_total_mb:.0f} MB GPU. "
        f"Full training will very likely OOM under augmentation + val. Either "
        f"drop CHUNK_POSITIONS to 1 or move to a larger GPU."
    )
del _model, _x, _y, _loss, _opt, _scaler
gc.collect(); torch.cuda.empty_cache()

# ── (b) 2-EPOCH SMOKE (real cache, real manifest, real trainer) ────────
os.makedirs(OUTPUT_DIR_SMOKE, exist_ok=True)
_smoke_cmd = [
    sys.executable, "-u",
    os.path.join(CODE_DIR, "training", "train_centralized_correct.py"),
    "--model",              MODEL_NAME,
    "--data_root",          DATA_ROOT,
    "--output_dir",         OUTPUT_DIR_SMOKE,
    "--epochs",             "2",
    "--batch_size",         str(BATCH_SIZE),
    "--num_workers",        "0",
    "--patch_size",         str(SMOKE_PATCH_SIZE[0]), str(SMOKE_PATCH_SIZE[1]), str(SMOKE_PATCH_SIZE[2]),
    "--lr",                 str(LEARNING_RATE),
    "--warmup_epochs",      "0",
    "--seed",               str(SEED),
    "--preprocessed_cache_dir", PREPROCESSED_CACHE_DIR,
    "--split_manifest",     MANIFEST_CSV,
    "--val_every",          "1",
    "--val_overlap",        str(VAL_OVERLAP),
    "--val_sw_batch_size",  str(VAL_SW_BATCH_SIZE),
    "--checkpoint_every",   "1",
    "--smoke_test",
]
assert "--deep_supervision" not in _smoke_cmd, "continuous is DS-off by contract"
print("\n(b) SMOKE command:\n  " + " ".join(_smoke_cmd))
_t0 = time.time()
_r = subprocess.run(_smoke_cmd, capture_output=False)
_smoke_s = time.time() - _t0
if _r.returncode != 0:
    raise SystemExit(f"CONTINUOUS GATE: smoke training exited {_r.returncode}")
print(f"(b) smoke wall  : {_smoke_s:.1f}s   -> pipeline healthy")

# ── Write the marker so cell 6 can proceed ─────────────────────────────
os.makedirs(OUTPUT_DIR_FULL, exist_ok=True)
with open(marker_path, "w") as _f:
    json.dump({
        "gpu_name":            torch.cuda.get_device_name(0),
        "gpu_total_mb":        _gpu_total_mb,
        "train_step_ms":       _step_s * 1e3,
        "train_step_peak_mb":  _peak_mb,
        "train_step_headroom_mb": _headroom_mb,
        "smoke_wall_seconds":  _smoke_s,
        "batch_size":          BATCH_SIZE,
        "patch_size":          list(PATCH_SIZE),
        "smoke_patch_size":    list(SMOKE_PATCH_SIZE),
        "manifest_sha256":     MANIFEST_SHA256,
        "arch_kind":           ARCH_KIND,
        "model_name":          MODEL_NAME,
    }, _f, indent=2)
print(f"\nCONTINUOUS GATE MARKER written : {marker_path}")
print("Cell 6 (FULL TRAINING) will now accept this run.")


In [ ]:
# ── 6. FULL TRAINING — 100 ep, formal val every 5 ep, manifest-driven ─
# On a mid-training Kaggle disconnect, come back and re-run this cell after
# setting RESUME_INPUT_DIR + RESUME_FROM in CONFIG (see cell 3).
import os, sys, subprocess

# Continuous variant: hard gate on the benchmark + 2-ep smoke marker.
if ARCH_KIND == "continuous":
    _marker = os.path.join(OUTPUT_DIR_FULL, "continuous_smoke_ok.json")
    if not os.path.exists(_marker):
        raise SystemExit(
            "FULL TRAINING blocked: this is the continuous notebook and the "
            "GPU benchmark + 2-epoch smoke has not been recorded. Run the "
            "cell titled 'CONTINUOUS SMOKE + GPU BENCHMARK' immediately "
            "above this one; it writes " + _marker + " on success. Do NOT "
            "delete that file to bypass the gate — the smoke exists to catch "
            "OOM / wall-time issues on the current Kaggle GPU."
        )

_cmd = [
    sys.executable, "-u",
    os.path.join(CODE_DIR, "training", "train_centralized_correct.py"),
    "--model",              MODEL_NAME,
    "--data_root",          DATA_ROOT,
    "--output_dir",         OUTPUT_DIR_FULL,
    "--epochs",             str(EPOCHS),
    "--batch_size",         str(BATCH_SIZE),
    "--num_workers",        str(NUM_WORKERS),
    "--patch_size",         str(PATCH_SIZE[0]), str(PATCH_SIZE[1]), str(PATCH_SIZE[2]),
    "--lr",                 str(LEARNING_RATE),
    "--warmup_epochs",      str(WARMUP_EPOCHS),
    "--seed",               str(SEED),
    "--k_att_temp_start",   str(K_ATT_TEMP_START),
    "--k_att_temp_end",     str(K_ATT_TEMP_END),
    "--k_att_anneal_epochs",str(K_ATT_ANNEAL_EPOCHS),
    "--preprocessed_cache_dir", PREPROCESSED_CACHE_DIR,
    "--split_manifest",     MANIFEST_CSV,
    "--val_every",          str(VAL_EVERY),
    "--val_overlap",        str(VAL_OVERLAP),
    "--val_sw_batch_size",  str(VAL_SW_BATCH_SIZE),
    "--checkpoint_every",   str(CHECKPOINT_EVERY),
]
if DEEP_SUPERVISION:
    _cmd.append("--deep_supervision")
if IS_RESUME_MODE:
    _resume_path = os.path.join(OUTPUT_DIR_FULL, os.path.basename(RESUME_FROM))
    assert os.path.isfile(_resume_path), f"resume mode but {_resume_path} missing"
    _cmd += ["--resume", _resume_path]
    print(f"RESUME: --resume {_resume_path} appended")

# Contract checks on the argv.
assert "--split_manifest" in _cmd
if DEEP_SUPERVISION:
    assert "--deep_supervision" in _cmd
else:
    assert "--deep_supervision" not in _cmd
assert "--val_every" in _cmd and _cmd[_cmd.index("--val_every") + 1] == "5"

print("FULL TRAINING command:\n  " + " ".join(_cmd))
print("=" * 60, flush=True)
proc = subprocess.Popen(_cmd, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, bufsize=0)
while True:
    chunk = proc.stdout.read(512)
    if not chunk:
        break
    sys.stdout.write(chunk.decode("utf-8", errors="replace"))
    sys.stdout.flush()
proc.wait()
if proc.returncode != 0:
    raise SystemExit(f"training exited nonzero ({proc.returncode})")


In [ ]:
# ── 7. STANDALONE FORMAL VALIDATION on best_model.pth ─────────────────
# Read-only formal evaluation. Enforces --require_manifest_checksum so a
# ckpt trained on a different split cannot be scored here by accident.
# Result JSON is labelled as formal validation on the fixed 306-patient
# default validation set — NEVER as 'test'.
import os, sys, subprocess

best_ckpt = os.path.join(OUTPUT_DIR_FULL, "best_model.pth")
if not os.path.exists(best_ckpt):
    print(f"(no best_model.pth at {best_ckpt}; skipping)")
else:
    out_json = os.path.join(OUTPUT_DIR_FULL, "formal_val_bestmodel.json")
    _cmd = [
        sys.executable, "-u",
        os.path.join(CODE_DIR, "training", "evaluate_correct_checkpoint.py"),
        "--checkpoint",              best_ckpt,
        "--data_root",               DATA_ROOT,
        "--preprocessed_cache_dir",  PREPROCESSED_CACHE_DIR,
        "--patch_size",              str(PATCH_SIZE[0]), str(PATCH_SIZE[1]), str(PATCH_SIZE[2]),
        "--overlap",                 str(VAL_OVERLAP),
        "--sw_batch_size",           str(VAL_SW_BATCH_SIZE),
        "--num_workers",             str(NUM_WORKERS),
        "--split_manifest",          MANIFEST_CSV,
        "--split_partition",         "val",
        "--require_manifest_checksum", MANIFEST_SHA256,
        "--per_collection",
        "--out",                     out_json,
    ]
    print("FORMAL VALIDATION command:\n  " + " ".join(_cmd))
    r = subprocess.run(_cmd, capture_output=False)
    if r.returncode != 0:
        raise SystemExit(f"formal validation failed ({r.returncode})")

    import json
    with open(out_json) as f:
        res = json.load(f)
    print()
    print("=" * 60)
    print("Formal validation metrics on the fixed 306-patient default validation set.")
    print(f"  n_cases      : {res['n_cases']}")
    print(f"  Dice (mean)  : {res['dice']:.4f}")
    print(f"  Dice (median): {res['dice_median']:.4f}")
    print(f"  Dice (std)   : {res['dice_std']:.4f}")
    print(f"  IoU / Sens   : {res['iou']:.4f} / {res['sensitivity']:.4f}")
    print(f"  manifest SHA : {MANIFEST_SHA256}")
    print(f"  json         : {out_json}")
    print("=" * 60)


In [ ]:
# ── 8. INTERRUPTED-VALIDATION RECOVERY ────────────────────────────────
# If Kaggle disconnected DURING the every-5 formal validation, the epoch's
# training already finished (last_checkpoint.pth was written before val
# started). Rather than re-training that epoch, flip RUN_RECOVERY = True
# below and re-run only this cell — it will:
#   1) formally evaluate last_checkpoint.pth (or a specified periodic ckpt)
#      against the 306-patient default validation set;
#   2) write the formal-val JSON to OUTPUT_DIR_FULL;
#   3) append the metrics into train_log.json;
#   4) update best_model.pth if the recovered Dice beats the checkpoint's
#      self-reported best_dice.
# After the recovery finishes, re-launch cell 6 with RESUME_* set in
# CONFIG to continue training from the next epoch.
RUN_RECOVERY = False
RECOVERY_CKPT_NAME = "last_checkpoint.pth"   # or "checkpoint_epoch045.pth" etc.

import os, sys, subprocess, json, shutil, tempfile
import torch

if not RUN_RECOVERY:
    print("RUN_RECOVERY=False — skipping. Flip to True and re-run only this "
          "cell to recover from a disconnect during validation.")
else:
    ckpt_path = os.path.join(OUTPUT_DIR_FULL, RECOVERY_CKPT_NAME)
    if not os.path.exists(ckpt_path):
        raise SystemExit(f"RECOVERY: no such checkpoint: {ckpt_path}")

    # Sanity: the ckpt must belong to THIS notebook's default-cache run.
    _c = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    _sid = (_c.get("split_identity") or {})
    _aid = (_c.get("arch_identity")  or {})
    if _sid.get("split_kind") != "default_cache":
        raise SystemExit(
            f"RECOVERY: ckpt split_kind={_sid.get('split_kind')!r}; refusing to "
            f"recover a non-default-cache checkpoint under this notebook."
        )
    if _sid.get("split_manifest_sha256") != MANIFEST_SHA256:
        raise SystemExit(
            f"RECOVERY: ckpt manifest SHA {_sid.get('split_manifest_sha256')!r} "
            f"!= this notebook's {MANIFEST_SHA256!r}"
        )
    if _aid.get("model_name") != MODEL_NAME:
        raise SystemExit(f"RECOVERY: model_name mismatch (ckpt={_aid.get('model_name')!r} vs {MODEL_NAME!r})")

    _completed_ep = int(_c.get("epoch", -1))
    print(f"RECOVERY: ckpt completed epoch = {_completed_ep}")
    print(f"          ckpt best_dice(self) = {float(_c.get('best_dice', 0.0)):.4f}")

    out_json = os.path.join(OUTPUT_DIR_FULL,
                            f"formal_val_recovery_ep{_completed_ep+1:03d}.json")
    _cmd = [
        sys.executable, "-u",
        os.path.join(CODE_DIR, "training", "evaluate_correct_checkpoint.py"),
        "--checkpoint",              ckpt_path,
        "--data_root",               DATA_ROOT,
        "--preprocessed_cache_dir",  PREPROCESSED_CACHE_DIR,
        "--patch_size",              str(PATCH_SIZE[0]), str(PATCH_SIZE[1]), str(PATCH_SIZE[2]),
        "--overlap",                 str(VAL_OVERLAP),
        "--sw_batch_size",           str(VAL_SW_BATCH_SIZE),
        "--num_workers",             str(NUM_WORKERS),
        "--split_manifest",          MANIFEST_CSV,
        "--split_partition",         "val",
        "--require_manifest_checksum", MANIFEST_SHA256,
        "--per_collection",
        "--out",                     out_json,
    ]
    print("RECOVERY eval command:\n  " + " ".join(_cmd))
    r = subprocess.run(_cmd, capture_output=False)
    if r.returncode != 0:
        raise SystemExit(f"recovery evaluation failed ({r.returncode})")

    with open(out_json) as f:
        res = json.load(f)
    rec_dice = float(res["dice"])
    rec_iou  = float(res["iou"])
    rec_sens = float(res["sensitivity"])
    print(f"\nRECOVERY formal metrics (n={res['n_cases']}): "
          f"Dice={rec_dice:.4f}  IoU={rec_iou:.4f}  Sens={rec_sens:.4f}")

    # 3) fold into train_log.json
    tl_path = os.path.join(OUTPUT_DIR_FULL, "train_log.json")
    log = []
    if os.path.exists(tl_path):
        with open(tl_path) as f:
            log = json.load(f)
    _entry_idx = None
    for i, e in enumerate(log):
        if e.get("epoch") == _completed_ep + 1:
            _entry_idx = i; break
    _fields = {
        "val_dice":        rec_dice,
        "val_iou":         rec_iou,
        "val_sensitivity": rec_sens,
        "val_n_cases":     res["n_cases"],
        "val_overlap":     VAL_OVERLAP,
        "val_source":      "recovery",
    }
    if _entry_idx is None:
        log.append({"epoch": _completed_ep + 1, **_fields})
    else:
        log[_entry_idx].update(_fields)

    _fd, _tmp = tempfile.mkstemp(prefix=".tmp_train_log.", suffix=".json",
                                 dir=OUTPUT_DIR_FULL)
    os.close(_fd)
    with open(_tmp, "w") as f:
        json.dump(log, f, indent=2, default=str)
    os.replace(_tmp, tl_path)
    print(f"train_log.json updated: {tl_path}")

    # 4) update best_model.pth if this recovered Dice beats ckpt.best_dice
    if rec_dice > float(_c.get("best_dice", 0.0)):
        _c["best_dice"] = rec_dice
        _c["train_log"] = log
        best_dst = os.path.join(OUTPUT_DIR_FULL, "best_model.pth")
        _fd, _tmp = tempfile.mkstemp(prefix=".tmp_best_model.", suffix=".pth",
                                     dir=OUTPUT_DIR_FULL)
        os.close(_fd)
        torch.save(_c, _tmp)
        os.replace(_tmp, best_dst)
        # last_checkpoint.pth mirrors the update.
        _fd, _tmp = tempfile.mkstemp(prefix=".tmp_last_checkpoint.", suffix=".pth",
                                     dir=OUTPUT_DIR_FULL)
        os.close(_fd)
        torch.save(_c, _tmp)
        os.replace(_tmp, os.path.join(OUTPUT_DIR_FULL, "last_checkpoint.pth"))
        print(f"best_model.pth UPDATED with recovered Dice {rec_dice:.4f}")
    else:
        print(f"recovered Dice {rec_dice:.4f} did not beat ckpt best_dice "
              f"{float(_c.get('best_dice', 0.0)):.4f} — best_model.pth left untouched.")


In [ ]:
# ── 9. DOWNLOAD LINKS ──────────────────────────────────────────────────
# List downloadable artifacts so the user can grab them before the Kaggle
# session dies. Per project discipline, download best_model.pth as soon as
# it lands, not at end-of-run.
import os
if os.path.isdir(OUTPUT_DIR_FULL):
    for name in sorted(os.listdir(OUTPUT_DIR_FULL)):
        full = os.path.join(OUTPUT_DIR_FULL, name)
        size_mb = os.path.getsize(full) / 1e6 if os.path.isfile(full) else 0
        print(f"  {size_mb:8.2f} MB   {full}")
else:
    print(f"(no output dir yet: {OUTPUT_DIR_FULL})")
